# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata and print high-level info
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Access top-level metadata fields
dataset_name = metadata.name
dataset_description = metadata.description
dataset_authors = getattr(metadata, 'author', None)
dataset_version = getattr(metadata, 'version', None)

print(f"Dataset: {dataset_name}\nDescription: {dataset_description}\nVersion: {dataset_version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by @id
print("Record Sets in the dataset:")
all_record_sets = list(dataset.record_sets.keys())
for idx, rset_id in enumerate(all_record_sets):
    record_set = dataset.record_sets[rset_id]
    print(f"{idx+1}. @id: {rset_id} | Name: {getattr(record_set, 'name', '')}")
    # List fields in this record set
    if hasattr(record_set, 'fields'):
        print("    Fields:")
        for field in record_set.fields:
            print(f"      - @id: {field['@id']}, name: {field.get('name', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Based on the overview, select the primary record set containing survey data.
# For this dataset, the main record set `@id` is usually 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/recordSet/survey', but let's list all and select the first with data.

record_set_ids = list(dataset.record_sets.keys())
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display available DataFrames
print("Extracted DataFrames for record sets with data:")
for key, df in dataframes.items():
    print(f"@id: {key} | shape: {df.shape}")

# For exploration, pick the first non-empty record set
main_record_set_id = next(iter(dataframes.keys()))
main_df = dataframes[main_record_set_id]

print(f"\nColumns in record set @{main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, let's pick a numeric field from the main_df (e.g., phq9_score or gad7_score if present)
# List all numeric columns
numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields detected:", numeric_fields)

# Pick 'phq9_score' if in data, else pick the first numeric field
if 'phq9_score' in main_df.columns:
    numeric_field = 'phq9_score'
elif 'gad7_score' in main_df.columns:
    numeric_field = 'gad7_score'
elif numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = None

if numeric_field:
    threshold = 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by another field - e.g., 'gender' or the first string column
    group_field = None
    for col in main_df.columns:
        if main_df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if a suitable categorical field is found
    if group_field:
        plt.figure(figsize=(8,6))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded the FAIR² Mental Health Survey Dataset using its Croissant schema, explored its record sets and fields (referencing all with their canonical `@id` values), and performed basic exploratory data analysis, including filtering, normalization, aggregation, and visualization.

This structured approach demonstrates how `mlcroissant` enables dataset introspection and analysis using schema metadata for reliable entity reference, supporting FAIR data principles in health data research for Africa.
